# Notebook demo: build & run the Dockerized JupyterLab for this repo

This notebook describes how to build and run the `notebooks` Docker image and Docker Compose stack included with this repository, how to execute notebooks headlessly, and how to make iterative code updates visible inside a running JupyterLab instance.

The steps below assume you are in the repository root and have Docker Desktop (or a compatible Docker engine) available.

## 1) What the Docker artifacts provide

- `Dockerfile.notebooks` — multi-stage image that installs your app dependencies and notebook runtime (JupyterLab + papermill).
- `docker-compose.notebooks.yml` — compose file that defines the `notebooks` service and a `qdrant` service for RAG demos. It maps port 8888 on the host to the container's JupyterLab server.
- `requirements-notebook.txt` — dependencies required for running notebooks and papermill.
- `run_notebooks.sh` — optional entrypoint that can run papermill to execute notebooks on startup and then start JupyterLab.

Why use this image? It gives a reproducible environment to run and test the notebooks without polluting your host Python environment.

## 2) Build and run (recommended quick start)

From the repository root run the following in your terminal. These commands build the image and bring up the compose stack (the `qdrant` service is optional but included for RAG demos).

```bash
# from repo root
docker compose -f docker-compose.notebooks.yml up --build
```

After the stack starts, open JupyterLab in your browser at http://localhost:8888 (no token in the demo image).

## 3) How code changes in the repo become visible in JupyterLab

There are two common workflows depending on whether you prefer quick iteration (bind-mounts) or a fully reproducible image build:

1) Bind-mount (fast iteration) — (the compose file by default mounts the repo into `/work`):

- Pros: change files on your host; the running Jupyter server immediately sees edits to Python modules and notebook files (no rebuild).
- Cons: Python modules already imported in the running kernel won't be reloaded automatically; kernel restart may be needed to pick up module-level changes.

How to make updates visible when using bind-mounts:
- Edit the file on your host (VS Code).
- In JupyterLab, re-open the notebook or the file tree (it should reflect edits automatically).
- For Python module changes (e.g., `app/gradio_main.py`), restart the notebook kernel or run `%reload_ext autoreload` and `%autoreload 2` in a cell to auto-reload modules (example below).

Example to enable autoreload in a notebook cell:
```python
%load_ext autoreload
%autoreload 2
```

2) Rebuild the image (reproducible) — when you've changed dependencies or want a repeatable snapshot:

- Rebuild the image and recreate the container. This ensures the Python environment inside the container matches the repository snapshot at build time.
- Commands:
```bash
# rebuild and replace the running container
docker compose -f docker-compose.notebooks.yml up --build
# OR to force recreate even without build changes
docker compose -f docker-compose.notebooks.yml up --build --force-recreate --renew-anon-volumes
```

Note: If you rebuild but keep the repo mounted as a volume, code files still come from the host; rebuilding is primarily needed when you changed `requirements` or the base image.

## 4) How to see notebook additions/edits without restarting the container

- Notebook files (`docs/*.ipynb`) are read from the mounted repo and appear immediately in the Jupyter file browser when saved on the host.
- If you add a new notebook file from the host, refresh the Jupyter file browser pane (or click the refresh icon) and the new file will appear.
- To run a newly added notebook headlessly from the container you can exec into the running container and run papermill or nbconvert; example below.

Example: run a notebook inside the running container (replace <container> with the container name):
```bash
# list containers
docker ps
# run papermill inside the notebooks container (writes output to notebook_outputs/)
docker exec -it <notebooks_container_name> bash -lc "papermill docs/architecture_and_workflow_overview.ipynb notebook_outputs/architecture_run.ipynb"
```

If you prefer automation, set `RUN_NOTEBOOKS=true` in the `docker-compose.notebooks.yml` environment section and the container's entrypoint will execute notebooks via `run_notebooks.sh` on startup (if present).

## 5) Quick troubleshooting

- Permission errors writing to a runtime directory: the image creates a non-root user and pre-creates `$HOME/.local/share/jupyter/runtime`. If you see PermissionError, ensure your Dockerfile sets `HOME` and the runtime path is owned by the container user (this repo's `Dockerfile.notebooks` already does this).
- Port conflict on 8888 or 6333: change the host-side mapping in `docker-compose.notebooks.yml` (e.g., `6334:6333` for qdrant) or stop the process listening on the host port.
- Kernel not picking up updated modules: restart the kernel from the Kernel menu or enable `%autoreload` as shown above.

## 6) Summary: when to rebuild vs. restart vs. reload

- Code edits (Python modules, notebook text): typically kernel restart or autoreload suffices — no rebuild needed when using a bind-mount.
- New/deleted notebook files: appear immediately in the file browser; refresh if necessary.
- Dependency changes (`requirements.txt`): rebuild the image and recreate the container.
- Permission / runtime path errors: fix by ensuring the container user has a writable HOME and runtime paths (already handled in `Dockerfile.notebooks`).

---

If you'd like, I can append a runnable cell that performs a quick import-smoke test from inside the notebook (it will try to import `app.gradio_main`, `app.rag`, `app.vector_store`, and `scripts.clients.complete` and print results). Tell me if you want that added and whether to mark the todo completed.

## 7) Screenshots & capturing the UI state for docs/PRs

When documenting the UI or opening a PR, it's helpful to capture screenshots from JupyterLab or your browser. Here are a few tips:

- Use the browser's native screenshot tool or a dedicated tool (macOS: Cmd+Shift+4, Windows: Snipping Tool).
- Capture the Jupyter file browser and the running terminal/kernel indicator to show the state of the environment.
- For reproducible images in docs, capture the notebook cell outputs (use the JupyterLab 'Save Notebook As' after running to preserve outputs).
- If you want automated screenshots during CI, consider a small Playwright or Puppeteer script that opens the notebook URL and captures a page screenshot after the server is ready. See CI example below.

## 8) CI integration (GitHub Actions) - run notebooks and capture artifacts

Below is a minimal GitHub Actions workflow that builds the `notebooks` image, runs it in the runner (service container), executes a notebook via papermill, and uploads the executed notebook as an artifact. Add this to `.github/workflows/notebooks-ci.yml` in your repo.

```yaml
name: Notebooks CI
on: [push, pull_request]
jobs:
  run-notebooks:
    runs-on: ubuntu-latest
    services:
      qdrant:
        image: qdrant/qdrant:latest
        ports: [6333:6333]
    steps:
      - uses: actions/checkout@v4
      - name: Build notebooks image
        run: docker build -f Dockerfile.notebooks -t repo-notebooks:ci .
      - name: Run notebooks container in background
        run: docker run -d --name ci_notebooks -v ${{ github.workspace }}:/work repo-notebooks:ci sleep infinity
      - name: Execute notebook
        run: docker exec ci_notebooks bash -lc "papermill docs/architecture_and_workflow_overview.ipynb notebook_outputs/ci_architecture.ipynb"
      - name: Upload artifact
        uses: actions/upload-artifact@v4
        with:
          name: executed-notebook
          path: notebook_outputs/ci_architecture.ipynb
      - name: Tear down
        if: always()
        run: docker rm -f ci_notebooks || true
```

Notes: the workflow above runs the notebooks container in the runner and executes one notebook using papermill. You can expand it to run multiple notebooks or install additional test dependencies as needed.

## 9) Automated papermill run-on-start (run_notebooks.sh example)

If you want the container to execute notebooks when it starts (headless execution), include a `run_notebooks.sh` script like the one below and set `RUN_NOTEBOOKS=true` in the compose environment. The entrypoint will run the script which executes a set of notebooks and then (optionally) starts JupyterLab so you can inspect results.

Example `run_notebooks.sh`:
```bash
#!/usr/bin/env bash
set -euo pipefail
mkdir -p /work/notebook_outputs
if [ "${RUN_NOTEBOOKS:-false}" = "true" ]; then
  echo 'Running notebooks via papermill...'
  papermill docs/architecture_and_workflow_overview.ipynb /work/notebook_outputs/architecture_run.ipynb || true
  papermill docs/notebooks_docker_demo.ipynb /work/notebook_outputs/docker_demo_run.ipynb || true
  echo 'Notebooks executed; outputs are in /work/notebook_outputs/'
fi
# Start JupyterLab (keep the container alive)
exec jupyter lab --ip=0.0.0.0 --port=8888 --no-browser --NotebookApp.token=''
```

To wire this up, ensure your `Dockerfile.notebooks` copies `run_notebooks.sh` and sets it executable, or mount it from the host and update the CMD in the Dockerfile to run the script as the container's command.

## 10) Example: small Playwright script to capture a JupyterLab screenshot

If you want an automated screenshot of the JupyterLab landing page in CI, the following Node.js Playwright snippet opens the Jupyter URL and saves a screenshot. You can run it in a GitHub Actions step after the container is ready.

```javascript
const { chromium } = require('playwright');
(async () => {
  const browser = await chromium.launch();
  const page = await browser.newPage();
  await page.goto('http://localhost:8888');
  await page.screenshot({ path: 'jupyterlab_screenshot.png', fullPage: true });
  await browser.close();
})();
```

Note: in CI, you may need to run Playwright with a headless browser and ensure the runner can reach the container port (use `services` or run the container in the same network).